In [ ]:
import pandas as pd

csv_path = "../../02_processed_data/FINAL_CLIMATE_FINANCIAL_DATA.csv"

# low_memory=False avoids the mixed-type warning during chunked inference
df = pd.read_csv(csv_path, low_memory=False)

print("Shape:", df.shape)

# show all columnsv
print("\nColumns:")
print(df.columns.tolist())

# show dtypes
print("\nDtypes:")
print(df.dtypes)

# quick missingness overview
missing = df.isna().mean().sort_values(ascending=False)
print("\nTop missingness (fraction):")
print(missing.head(15))


In [2]:
import random
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# 0) Repro + device
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# 1) Load
# -----------------------------
csv_path = "../../02_processed_data/FINAL_CLIMATE_FINANCIAL_DATA.csv"
df = pd.read_csv(csv_path, low_memory=False)

# -----------------------------
# 2) Fix types
# -----------------------------
# real_gdp is object -> numeric
df["real_gdp"] = pd.to_numeric(df["real_gdp"], errors="coerce")

# If you want to confirm conversion:
print("real_gdp dtype:", df["real_gdp"].dtype, "missing:", df["real_gdp"].isna().mean())

# -----------------------------
# 3) Target + split by year
# -----------------------------
target = "hpi_change"

# drop rows missing target
df = df[df[target].notna()].copy()

train_df = df[(df["yr"] >= 2000) & (df["yr"] <= 2019)].copy()
test_df  = df[df["yr"] == 2020].copy()

print("Train shape:", train_df.shape, "years:", train_df["yr"].min(), "-", train_df["yr"].max())
print("Test shape :", test_df.shape, "years:", test_df["yr"].unique())

if len(test_df) == 0:
    raise ValueError("No 2020 rows found. Check df['yr'].")

# -----------------------------
# 4) Features
#   - numeric only
#   - exclude identifiers + target + yr
#   - exclude state (object)
# -----------------------------
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()

exclude_cols = {
    target,
    "yr",
    "Unnamed: 0",     # row index
    "county_fips5",   # identifier
    "county_fips"     # identifier (float)
}

feature_cols = [c for c in numeric_cols if c not in exclude_cols]

print("\nNum features:", len(feature_cols))
print("Feature cols:", feature_cols)

X_train_full = train_df[feature_cols].values.astype(np.float32)
y_train_full = train_df[target].values.astype(np.float32).reshape(-1, 1)

X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df[target].values.astype(np.float32).reshape(-1, 1)

# -----------------------------
# 5) Validation split from training years
# -----------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=SEED
)

# -----------------------------
# 6) Impute + scale (fit only on train)
# -----------------------------
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)

# -----------------------------
# 7) Torch dataset
# -----------------------------
class TabularAs1DDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float().unsqueeze(1)  # (N, 1, L)
        self.y = torch.from_numpy(y).float()               # (N, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabularAs1DDataset(X_train_scaled, y_train)
val_ds   = TabularAs1DDataset(X_val_scaled,   y_val)
test_ds  = TabularAs1DDataset(X_test_scaled,  y_test)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

# -----------------------------
# 8) Model
# -----------------------------
class HPI_CNN(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 1, seq_len)
            out = self.conv(dummy)
            flat_size = out.view(1, -1).shape[1]

        self.fc = nn.Sequential(
            nn.Linear(flat_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

seq_len = X_train_scaled.shape[1]
model = HPI_CNN(seq_len).to(device)
print(model)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# -----------------------------
# 9) Train + early stopping
# -----------------------------
num_epochs = 20
patience = 4
min_delta = 1e-5

best_val = float("inf")
best_state = copy.deepcopy(model.state_dict())
no_improve = 0

for epoch in range(1, num_epochs + 1):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            val_losses.append(loss.item())

    train_mse = float(np.mean(train_losses))
    val_mse = float(np.mean(val_losses))
    print(f"Epoch {epoch:03d} | train MSE {train_mse:.6f} | val MSE {val_mse:.6f}")

    if val_mse < best_val - min_delta:
        best_val = val_mse
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch} (best val MSE {best_val:.6f})")
            break

model.load_state_dict(best_state)

# -----------------------------
# 10) Evaluate on 2020 holdout
# -----------------------------
model.eval()
preds, trues = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        yhat = model(xb).cpu().numpy()
        preds.append(yhat)
        trues.append(yb.numpy())

y_pred = np.vstack(preds).reshape(-1)
y_true = np.vstack(trues).reshape(-1)

mse  = mean_squared_error(y_true, y_pred)
rmse = float(np.sqrt(mse))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

eps = 1e-8
mape = float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100.0)
smape = float(np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100.0)

print("\n===== 2020 HOLDOUT METRICS (target=hpi_change) =====")
print(f"N (2020) : {len(y_true)}")
print(f"MSE      : {mse:.6f}")
print(f"RMSE     : {rmse:.6f}")
print(f"MAE      : {mae:.6f}")
print(f"R^2      : {r2:.6f}")
print(f"MAPE %   : {mape:.3f}")
print(f"sMAPE %  : {smape:.3f}")

# -----------------------------
# 11) Save predictions
# -----------------------------
pred_df = test_df[["county_fips5", "state", "yr", target]].copy()
pred_df["y_pred"] = y_pred
pred_df["residual"] = pred_df[target] - pred_df["y_pred"]

pred_df.to_csv("predictions_2020.csv", index=False)
print("\nSaved predictions_2020.csv")
print(pred_df.head())


Device: cpu
real_gdp dtype: float64 missing: 1.7469995283101274e-05
Train shape: (54455, 23) years: 2000 - 2019
Test shape : (2786, 23) years: [2020]

Num features: 17
Feature cols: ['index_nsa', 'flood_frequency', 'flood_property_damage', 'flood_duration_hours', 'heat_stress_index', 'drought_annual_mean_index', 'fire_frequency', 'fire_size', 'deaths_hurricane', 'injuries_hurricane', 'damage_hurricane', 'real_gdp', 'unemployment_rate', 'prop_rate', 'total_home_purchase', 'total_amt_purchase', 'total_population']
HPI_CNN(
  (conv): Sequential(
    (0): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): ReLU()
    (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)
Epoch 001 | train MSE 27.780763 